# TAC2017

11/04/2025

In [1]:
from lxml import etree
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1,2"
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import json
import re
from statistics import mean

## Summary Statistics

In [2]:
def load_xml(path: str) -> etree._Element:
    parser = etree.XMLParser(remove_blank_text=True, recover=True, huge_tree=True)
    return etree.parse(path, parser=parser).getroot()

In [5]:
def get_sections_and_mentions(root: etree._Element):
    """
    Return two dicts:
       sections: {section_id: section_name}
       mentions: list of (mention_id, section_id, type)
    """
    # gather sections
    sections = {}
    for sec in root.findall(".//Text//Section"):
        sid = sec.get("id")
        sname = sec.get("name")
        if sid:
            sections[sid] = sname

    mentions = []
    for m in root.findall(".//Mentions/Mention"):
        mid = m.get("id")
        sid = m.get("section")
        mtype = m.get("type")
        mentions.append((mid, sid, mtype))
    return sections, mentions

In [4]:
def summarize_tac2017_dir(xml_dir: str):
    """
    Traverse all XML files in xml_dir, compute:
      - total number of sections
      - sections with at least one ADR mention
      - sections without ADR mention
      - average ADR mentions per section
    """
    total_sections = 0
    sections_with_adr = 0
    sections_without_adr = 0
    adrs_per_section = []

    for fname in os.listdir(xml_dir):
        if not fname.lower().endswith(".xml"):
            continue
        path = os.path.join(xml_dir, fname)
        try:
            root = load_xml(path)
        except Exception as e:
            print(f"[WARN] Could not parse {fname}: {e}")
            continue

        sections, mentions = get_sections_and_mentions(root)
        # build map of section_id -> count of ADR mentions
        adr_counts = {sid: 0 for sid in sections.keys()}
        for _, sid, mtype in mentions:
            if mtype and mtype.lower() == "adversereaction" and sid in adr_counts:
                adr_counts[sid] += 1

        total_sections += len(sections)
        for sid, count in adr_counts.items():
            if count > 0:
                sections_with_adr += 1
            else:
                sections_without_adr += 1
            adrs_per_section.append(count)

    avg_adrs_per_section = mean(adrs_per_section) if adrs_per_section else 0.0

    summary = {
        "total_xml_files": len([f for f in os.listdir(xml_dir) if f.lower().endswith(".xml")]),
        "total_sections": total_sections,
        "sections_with_adr": sections_with_adr,
        "sections_without_adr": sections_without_adr,
        "avg_adrs_per_section": round(avg_adrs_per_section, 3)
    }
    return summary

In [6]:
xml_directory = 'data/tac2017/'
summary_stats = summarize_tac2017_dir(xml_directory)
print("TAC 2017 ADR Dataset Summary:")
for k, v in summary_stats.items():
    print(f"  {k}: {v}")

TAC 2017 ADR Dataset Summary:
  total_xml_files: 200
  total_sections: 476
  sections_with_adr: 472
  sections_without_adr: 4
  avg_adrs_per_section: 55.647


## Baseline Results

In [2]:
# run_baseline.py
import glob
from VaxMapper.src.baseline_adr_llm import run_zero_shot_on_file, load_inference_model, make_hf_llm_fn
from VaxMapper.src.eval_adr_spans import evaluate_files

In [3]:
INFERENCE_MODEL_ID = ["Qwen/Qwen2-7B-Instruct", "google/medgemma-27b-text-it"]

In [4]:
model, tokenizer = load_inference_model(INFERENCE_MODEL_ID[0])

Loading inference model Qwen/Qwen2-7B-Instruct...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Inference model Qwen/Qwen2-7B-Instruct loaded successfully.


In [ ]:
llm_fn = make_hf_llm_fn(
    model, tokenizer,
    max_new_tokens=768,
    temperature=0.0,
    top_p=1.0,
    repetition_penalty=1.0,
    stop_strings=None,
    enforce_json=True
)

In [6]:
test = "data/tac2017/ACTEMRA.xml"

In [7]:
result = run_zero_shot_on_file(test, llm_fn, 3800)

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


ValueError: invalid literal for int() with base 10: '5566,5612'

In [8]:
result

NameError: name 'result' is not defined

In [7]:
xml_dir = "data/tac2017/"
paths = sorted(glob.glob(f"{xml_dir}/*.xml"))

results = []
for p in paths:
    out = run_zero_shot_on_file(p, llm_fn=llm_fn, max_tokens=1800)
    results.append(out)

# Exact match
metrics_exact = evaluate_files(results, mode="exact")
print("Exact:", metrics_exact)

# Overlap ≥ 1 char (lenient)
metrics_overlap = evaluate_files(results, mode="overlap", min_overlap=1)
print("Overlap≥1:", metrics_overlap)


error: look-behind requires fixed-width pattern

In [ ]:
# parse the XML file in data/tac2017
directory = 'data/tac2017/'
for entry in os.listdir(directory):
    if entry.endswith('.xml'):
        tree = etree.parse(os.path.join(directory, entry))
        root = tree.getroot()
        drug_label = root.xpath('//Label[@drug]')[0].get("drug")
        for text_node in root.xpath('//Text/Section'):
            text_content = text_node.text
        for reaction in root.xpath('//Reactions/Reaction'):
            reaction_text = reaction.get('str')
            for normalization in reaction.xpath('./Normalization'):
                medra_pt_str = normalization.get('meddra_pt')
                meddra_pt_id = normalization.get('meddra_pt_id')

In [2]:
def load_inference_model(model_id):
    print(f"Loading inference model {model_id}...")
    tokenizer = AutoTokenizer.from_pretrained(
        model_id,
        use_fast=True,
        trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        trust_remote_code=True)
    print(f"Inference model {model_id} loaded successfully.")
    return model, tokenizer

In [32]:
inference_model_id = INFERENCE_MODEL_ID[1]
inf_model, inf_tokenizer = load_inference_model(inference_model_id)

Loading inference model google/medgemma-27b-text-it...


Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/11 [00:00<?, ?it/s]

Inference model google/medgemma-27b-text-it loaded successfully.


In [ ]:
torch.set_float32_matmul_precision('high')

In [33]:
def build_prompt(drug_label, adr_section):
    # --- System role ---
    sys_msg = ("""\
You are an expert biomedical natural language processing system analyzing FDA Structured Product Labels.
Your task is to perform zero-shot Named Entity Recognition (NER) to extract all Adverse Drug Reactions (ADRs)
from the following text snippet.

Instructions:
1. Only identify terms that represent an Adverse Drug Reaction or Event.
2. Ignore frequency modifiers or clinical context outside the core event.
3. Your output MUST be a valid JSON object containing a key named "extracted_adrs",
   which is a list of objects with the keys "term" and "source_section".
4. Do not include any other text, explanation, or code fences before or after the JSON."""

    )

    # --- User message ---
    user_msg = (f"""\
### SPL Adverse Reactions Extraction Task
Drug: {drug_label}
Adverse Reactions Text: {adr_section}"""
    )


    return [
        {"role": "system", "content": sys_msg},
        {"role": "user", "content": user_msg}
    ]


In [42]:
def run_adr_extraction_inference(prompt, inf_tokenizer, inf_model):

    # chat = build_prompt(drug_label, adr_section)
    text = inf_tokenizer.apply_chat_template(
        prompt,
        add_generation_prompt=True,
        tokenize=False
    )
    model_inputs = inf_tokenizer([text], return_tensors="pt", padding=True).to(inf_model.device)
    if inf_tokenizer.pad_token is None:
        inf_tokenizer.pad_token = inf_tokenizer.eos_token

    generated_ids = inf_model.generate(
        model_inputs.input_ids,
        pad_token_id=inf_tokenizer.pad_token_id,
        max_new_tokens=500,
        # repetition_penalty=1.1,
        # do_sample=False
    )
    generated_ids = [output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)]
    assistant_text = inf_tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()

    # Try to extract JSON from the response
    try:
        # First attempt: parse as-is
        return json.loads(assistant_text)
    except json.JSONDecodeError:
        # Second attempt: find JSON-like patterns with regex
        json_pattern = r'(\{.*\})'
        match = re.search(json_pattern, assistant_text, re.DOTALL)
        if match:
            try:
                return json.loads(match.group(1))
            except:
                pass
        
        # Fallback: return structured error response
        print(f"Failed to parse JSON from: {assistant_text}...")
        # return {
        #     "term": "NONE",
        #     "source_": "none",
        #     "justification": "Error parsing model output",
        #     "error": "JSONDecodeError"
        # }

    # return json.loads(assistant_text)

In [40]:
drug_label

'belviq'

In [43]:
prompt = build_prompt(drug_label, text_content)
run_adr_extraction_inference(prompt, inf_tokenizer, inf_model)

skipping cudagraphs due to skipping cudagraphs due to multiple devices: device(type='cuda', index=0), device(type='cuda', index=1)


Failed to parse JSON from: ```json
{
  "extracted_adrs": [
    {
      "term": "Serotonin Syndrome",
      "source_section": "5 WARNINGS AND PRECAUTIONS"
    },
    {
      "term": "Neuroleptic Malignant Syndrome",
      "source_section": "5 WARNINGS AND PRECAUTIONS"
    },
    {
      "term": "Valvular heart disease",
      "source_section": "5 WARNINGS AND PRECAUTIONS"
    },
    {
      "term": "Cognitive Impairment",
      "source_section": "5 WARNINGS AND PRECAUTIONS"
    },
    {
      "term": "disturbances in attention",
      "source_section": "5 WARNINGS AND PRECAUTIONS"
    },
    {
      "term": "memory disturbances",
      "source_section": "5 WARNINGS AND PRECAUTIONS"
    },
    {
      "term": "Psychiatric Disorders",
      "source_section": "5 WARNINGS AND PRECAUTIONS"
    },
    {
      "term": "euphoria",
      "source_section": "5 WARNINGS AND PRECAUTIONS"
    },
    {
      "term": "dissociation",
      "source_section": "5 WARNINGS AND PRECAUTIONS"
    },
    {
    

In [ ]:




# ---------- Example ----------
if __name__ == "__main__":
    xml_dir = "/path/to/TAC2017_XMLs"
    stats = summarize_tac2017_dir(xml_dir)
    print("TAC 2017 ADR Summary Statistics")
    for k, v in stats.items():
        print(f"{k:25s}: {v}")
